<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_3_panel_strike.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · 3 — The Wave of a Struck Panel (hyperbolic)

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L7.2 · Fundamental PDEs**

A tensioned panel is struck, and it rings. How does it move over the next three
periods? The wave equation answers it; you solve it by finite differences and
by a physics-informed network, and score both against the exact solution.

This notebook stands on its own: the die and the slot are the other two
notebooks of the set, and neither is needed here.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the exact solution, and finite differences on a mesh for an agreed 1 % | — |
| **5** | a PINN with the edges and the flat start built in, and the comparison | the trial function, the residual |
| **6** | what the comparison says | — |
| **7 – 8** | your report, and the mini project | your answers |

About fifteen minutes on a CPU, most of it training: a wave is the hardest of
the three for a network. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex07.2_outputs"                          # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

A square panel, tensioned and clamped at its edges, is struck across its face.
It starts flat and moving. How does it move over the next 20 ms - and how far
does it deflect?

## 2 · The data

| | |
|---|---|
| panel | 0.40 × 0.40 m, edges clamped: $u$ = 0 |
| wave speed | $c$ = 80 m/s, from the tension and the areal density |
| start | flat, $u = 0$, and moving at up to 0.5 m/s in the shape of the panel's first mode |
| window | 0 to 20 ms, about three periods of the 141 Hz fundamental |

## 3 · The physics

$$u_{tt} = c^2\,(u_{xx} + u_{yy}), \qquad u = 0 \text{ on the edges}, \qquad u(x, y, 0) = 0, \quad u_t(x, y, 0) = v_0(x, y)$$

The wave equation is **hyperbolic**: disturbances travel at speed $c$ and do
not fade. It needs **two** initial conditions - where the panel is and how fast
it moves - and an error made early is carried along, not damped. Struck in the
shape of its first mode, the panel moves in that mode only:

$$u = \frac{v_{\max}}{\omega}\,\sin\omega t\,\sin\tfrac{\pi x}{L}\sin\tfrac{\pi y}{L}, \qquad \omega = \sqrt2\,\pi c/L$$

which peaks at 0.56 mm. The cell below draws the centre's motion.

In [ ]:
L, T_END = pb.L_PANEL, pb.WAVE_T_END
U = pb.V_STRIKE / pb.OMEGA                                    # m: the peak deflection
print(f"fundamental {pb.OMEGA / (2 * np.pi):.2f} Hz, period {pb.wave_period()*1e3:.2f} ms, peak {U*1e3:.2f} mm")
t = np.linspace(0, T_END, 400)
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(t * 1e3, pb.wave_exact(np.full_like(t, L / 2), np.full_like(t, L / 2), t) * 1e3)
ax.set_xlabel("t  [ms]"); ax.set_ylabel("u at the centre  [mm]"); ax.set_title("the exact motion of the centre")
plt.tight_layout(); plt.show()

**What you should see.** A fundamental of 141.42 Hz, a period of 7.07 ms, a
peak of 0.56 mm, and nearly three swings of the centre.

---

## 4 · The reference, and finite differences

**The reference** is the exact solution above.

**Finite differences** step the wave explicitly: the five-point stencil in space
and the leapfrog in time, $u^{n+1} = 2u^n - u^{n-1} + (c\,\Delta t)^2\,\nabla_h^2 u^n$,
started with $u^1 = \Delta t\,v_0$. It is only stable if a step is shorter than
the time the wave takes to cross a cell - here half of it. **The agreed accuracy
is 1 % of the peak**, 5.6 µm, over the whole window.

In [ ]:
def fdm_panel(n):
    # n interior nodes a side, leapfrog at half the stability limit; the field at every step
    h = L / (n + 1); x = np.arange(1, n + 1) * h
    X, Y = np.meshgrid(x, x)
    I1, T1 = sp.identity(n), sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n))
    A = ((sp.kron(I1, T1) + sp.kron(T1, I1)) / h ** 2).tocsr()     # the five-point Laplacian
    steps = int(np.ceil(T_END / (0.5 * h / (pb.C_WAVE * np.sqrt(2))))); dt = T_END / steps
    u0 = np.zeros(n * n); u1 = dt * pb.wave_velocity(X, Y, 0.0).ravel()   # flat, and moving
    out = [u0, u1]
    for _ in range(steps - 1):
        u0, u1 = u1, 2 * u1 - u0 + (pb.C_WAVE * dt) ** 2 * (A @ u1)
        out.append(u1)
    return X, Y, np.linspace(0, T_END, steps + 1), np.array(out)

TARGET = 0.01 * U                                             # m: the agreed accuracy, 1 % of the peak
for n in (9, 19, 39, 79):
    X, Y, ts, V = fdm_panel(n)
    err = max(np.abs(V[k] - pb.wave_exact(X, Y, tk).ravel()).max() for k, tk in enumerate(ts))
    sec = middle(lambda: fdm_panel(n))
    print(f"  {n+2:3d} x {n+2} nodes, {len(ts)-1} steps: worst error {err*1e6:.2f} um ({100*err/U:.2f} % of the peak), "
          f"one run {sec*1e3:.0f} ms" + ("   <- meets 1 %" if err <= TARGET else ""))
N_FDM = 39                                                    # the coarsest mesh that meets it

**What you should see.** The error falling four times with every halving -
4.91, 1.22, 0.30 and 0.08 % of the peak - and 41 × 41 nodes with 453 steps
the first under 1 %, the whole window in about 10 ms.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(X, Y, \tau)$ - position over $L$, time over the 20 ms
window. A wave over three periods needs more of it than the die did: five layers
of 48, and 6000 collocation points.

**The edges and the flat start are built in**: the trial function
$\hat u = \tau\;16X(1-X)Y(1-Y)\;U\,\mathcal{N}$ is zero on every edge and at
$\tau = 0$. The **velocity** at the start cannot be built in the same way, so it
enters the loss: $\hat u_t(x, y, 0) = v_0$ at 400 points.

**Scaling decides whether it trains at all.** Over the window the panel's
acceleration is about $(\omega T)^2 \approx 350$ times its displacement, so the
residual is divided by $U\omega^2$ - otherwise it outweighs the velocity term,
and the network settles on the one motion that satisfies everything else: none.

**TODO 1** is the trial function; **TODO 2** the residual.

In [ ]:
class Panel(torch.nn.Module):                                 # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=48, n_layers=5)       # dense tanh network: X, Y, tau in
    def forward(self, p):
        X, Y, tau = p[:, :1], p[:, 1:2], p[:, 2:]
        # TODO 1: the trial function - zero on every edge and at tau = 0.
        #
        #   return U * tau * 16 * X * (1 - X) * Y * (1 - Y) * self.net(p)
        raise NotImplementedError("Write the trial function")

In [ ]:
C = pb.C_WAVE * T_END / L                                     # 4: the wave speed in scaled units
R_SCALE = U * (pb.OMEGA * T_END) ** 2                         # the size of u_tau_tau
# TODO 2: the residual of the wave equation, divided by R_SCALE.
#
#   def residual(model, p):
#       u = model(p)
#       return (d2(u, p, 2) - C ** 2 * (d2(u, p, 0) + d2(u, p, 1))) / R_SCALE
raise NotImplementedError("Write residual(model, p)")

In [ ]:
pts = to_tensor(interior_points(6000, ((0, 1), (0, 1), (0, 1)), method="lhs", seed=1), requires_grad=True)  # collocation points, differentiable
start = interior_points(400, ((0, 1), (0, 1)), method="lhs", seed=2)   # 400 points on the panel at the start
p0 = to_tensor(np.c_[start, np.zeros(len(start))], requires_grad=True)      # tau = 0, differentiable
v0 = to_tensor(pb.wave_velocity(start[:, 0] * L, start[:, 1] * L, 0.0).reshape(-1, 1))   # the strike, m/s
model = Panel().to(DEVICE)                                    # five layers of 48
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio

def loss():                                                   # what train_two_stage minimises
    ut = grad(model(p0), p0)[:, 2:] / T_END                   # the velocity at the start, by autograd, m/s
    return residual(model, pts).pow(2).mean() + ((ut - v0) / pb.V_STRIKE).pow(2).mean()

t0 = time.perf_counter()
train_two_stage(model, loss, adam_steps=6000, lbfgs_steps=250, lr=1e-3, report_every=0)   # Adam first, then L-BFGS
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
X, Y, ts, V = fdm_panel(N_FDM)                                # the finite-difference answer, every step
Pn = to_tensor(np.c_[X.ravel() / L, Y.ravel() / L])           # the same nodes for the network
def pinn_at(t):
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(torch.cat([Pn, torch.full((len(Pn), 1), t / T_END)], 1))).ravel()
e_fdm = np.array([np.abs(V[k] - pb.wave_exact(X, Y, tk).ravel()).max() for k, tk in enumerate(ts)])
e_pin = np.array([np.abs(pinn_at(tk) - pb.wave_exact(X, Y, tk).ravel()).max() for tk in ts])
t_fdm = middle(lambda: fdm_panel(N_FDM))
t_pin = middle(lambda: [pinn_at(tk) for tk in ts])
print(f"{'':28s}{'worst error':>16s}{'whole window':>15s}{'training':>10s}")
print(f"  {'finite differences':26s}{e_fdm.max()*1e6:9.2f} um {100*e_fdm.max()/U:4.2f} %{t_fdm*1e3:10.1f} ms{'-':>10s}")
print(f"  {'PINN':26s}{e_pin.max()*1e6:9.2f} um {100*e_pin.max()/U:4.2f} %{t_pin*1e3:10.1f} ms{train_time:8.0f} s")
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(ts * 1e3, 100 * e_fdm / U, label="finite differences, 41 x 41"); ax.plot(ts * 1e3, 100 * e_pin / U, label="PINN")
ax.axhline(1.0, color="k", lw=0.8, ls="--", label="agreed 1 %")
ax.set_xlabel("t  [ms]"); ax.set_ylabel("worst error, % of the peak"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex07.2_3_panel_strike_report.png"), dpi=150); plt.show()

**What you should see.** Finite differences at 1.71 µm, 0.30 % of the peak,
and the network at about 2.74 µm, 0.49 % - both within the agreed 1 % - with
both errors growing with time. Finite differences take about 10 ms and no
training; the network about 400 ms for the same 454 instants on the same
nodes, after about fourteen minutes of training.

---

## 6 · What the comparison says

* **Both meet the agreed accuracy, and both errors grow with time.** A wave
  carries what it is given: an error made early travels on instead of being
  damped. That is the hyperbolic type, and it is the opposite of the die.
* **The network needed five times the training of the die** for the same
  relative accuracy - more neurons, more points, more steps, and a residual
  scaled to the wave's own size before it would move at all.
* **Finite differences win this one clearly**: milliseconds and no training. A
  network earns its place on a wave only when it is asked something a mesh
  cannot answer cheaply - where the panel was struck, from a few sensors, is the
  mini project below.

---

## 7 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex07.2_3_panel_strike_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L7.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "What does 'hyperbolic' mean for the panel, and why does it need two initial conditions where the die needed one? (-> L7.2 Q9)": """
""",
    "Why do the errors of both methods grow with time here, when on the die they fell? (-> L7.2 Q9)": """
""",
    "Why was the start's velocity a loss term and not built in, and why did the residual need dividing by U omega^2? (-> L7.2 Q10)": """
""",
    "To conclude: which of the three benchmark types - parabolic, elliptic, hyperbolic - was hardest for the network, and what did finite differences need for it? (-> L7.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | worst error | whole window | training |", "|---|---|---|---|",
            f"| finite differences, 41 x 41 | {e_fdm.max()*1e6:.2f} um ({100*e_fdm.max()/U:.2f} %) | {t_fdm*1e3:.1f} ms | - |",
            f"| PINN, 5 x 48 | {e_pin.max()*1e6:.2f} um ({100*e_pin.max()/U:.2f} %) | {t_pin*1e3:.1f} ms | {train_time:.0f} s |"]
    out = ["# Ex_07.2 — The Wave of a Struck Panel", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex07.2_3_panel_strike_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.2_3_panel_strike_report.md into Ex07.2_3_panel_strike_report.pdf, with any figure
# saved as Ex07.2_3_panel_strike_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_3_panel_strike_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_3_panel_strike_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex07.2_3_panel_strike_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.2_3_panel_strike_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 8 · Mini project proposal

One problem grown out of this notebook, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP7.2C · Where was the panel struck?

**The problem.** The panel of this notebook was struck at a known place with a known speed. Turn it
round: a few sensors on the panel record its motion after a strike nobody saw.
From those records alone, find where it was struck, how hard and how wide -
and then the whole motion of the panel.

| | |
|---|---|
| panel | 0.40 × 0.40 m, wave speed 80 m/s, edges clamped, flat at the start |
| strike | a Gaussian of initial velocity: centre, width and peak unknown |
| records | displacement, 0 to 20 ms, 201 samples per sensor |
| case 1 | 6 sensors, noise 2 µm |
| case 2 | 4 sensors, noise 10 µm |
| case 3 | 3 sensors on the diagonal, at (0.1, 0.1), (0.2, 0.2), (0.3, 0.3) m, noise 2 µm |

**Why a plain PINN is not enough.** The initial condition is now the unknown, not the given, and a loss can
converge on a problem that does not have one answer. With too few sensors, or sensors placed symmetrically, different strikes give
the same records.

**What you build.** An inverse PINN: one network for the displacement $u(x, y, t)$ and one for
the initial velocity $v_0(x, y)$ - or its three parameters - fitted together
to the wave equation, the clamped edges, $u = 0$ at the start and the sensor
records.

**The ground truth you get.** `MP7.2B_readings.npz` to fit and `MP7.2B_truth.npz` to score against,
from the same script: the panel's motion is exact as a sum of its modes, taken
to 80 × 80 (20 × 20 already agrees to 0.01 µm).

**Worked example.** Case 1's strike is centred at (0.13, 0.27) m, 25 mm wide, peaking at
1.0 m/s. The panel's largest displacement is 0.23 mm; the sensors see up to
0.16 mm. In case 3 the strike and its mirror image in the diagonal give
readings that differ by 2e-19 m - nothing any sensor could tell apart.

**Requirements.**

* Case 1: the strike point within **10 mm**, its peak velocity within **10 %** and its width within **20 %**.
* Case 1: the displacement within **5 %** of its peak, 11 µm, everywhere over the 20 ms.
* Case 3: show two strikes that fit the records equally well, and say why.

**What you hand in.** The recovered strike against the true one in each case, the motion's error,
and how both depend on the number and placement of the sensors. Show the
placement that fails, and why.